In [1]:
from pathlib import Path
import sys
import warnings
import os
import gc
from copy import deepcopy

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.ops import MLP
import torch.optim as optim
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, StratifiedKFold
from torch.utils.data import Dataset, TensorDataset, DataLoader, Subset
import optuna
import joblib
from optuna.samplers import TPESampler

import random
import math
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
import anndata
from anndata import AnnData
import pickle

sys.path.insert(0, "../../")

import scgpt as scg
from scgpt.tokenizer import GeneVocab
from scgpt import logger

warnings.filterwarnings("ignore", category=ResourceWarning)

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    
seed = 13
seed_everything(seed)

/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/celltypeinitialized_comparison/../../scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/celltypeinitialized_comparison/../../scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [2]:
cell_embeddings_adata = anndata.read_h5ad("../Data Preprocessing/cell_embeddings_adata.h5ad")
combined_ephys_df = pd.read_pickle("../Data Preprocessing/combined_ephys_df.pkl")

scgpt_embeddings_tensor = torch.tensor(cell_embeddings_adata.obsm["X_scGPT"], dtype=torch.float32)
hvg_expression_tensor = torch.tensor(cell_embeddings_adata[:, cell_embeddings_adata.var["highly_variable"]].X, dtype=torch.float32)
ionchannel_expression_tensor = torch.tensor(cell_embeddings_adata[:, cell_embeddings_adata.var["Ion channel coding"]].X, dtype=torch.float32)
cell_type_tensor = torch.tensor(cell_embeddings_adata.obsm["OneHotCellType"], dtype=torch.float32)

ephys_tensor = torch.tensor(combined_ephys_df.drop(columns=["SpecimenID"]).values, dtype=torch.float32)

embeddings_withtype_tensor = torch.cat((scgpt_embeddings_tensor, cell_type_tensor), dim=1)
hvg_expression__withtype_tensor = torch.cat((hvg_expression_tensor, cell_type_tensor), dim=1)
ionchannel_expression_withtype_tensor = torch.cat((ionchannel_expression_tensor, cell_type_tensor), dim=1)

scgpt_dataset = TensorDataset(embeddings_withtype_tensor, ephys_tensor)
hvg_dataset = TensorDataset(hvg_expression__withtype_tensor, ephys_tensor)
ionchannel_dataset = TensorDataset(ionchannel_expression_withtype_tensor, ephys_tensor)
onehot_dataset = TensorDataset(cell_type_tensor, ephys_tensor)
datasets = {"onehot": onehot_dataset, "scgpt": scgpt_dataset, "hvg": hvg_dataset, "ion": ionchannel_dataset}

In [3]:
def initialize_weights(model, path_to_pretrained):
    model_sd = model.state_dict()
    pretrained_sd = torch.load(path_to_pretrained)

    for layer, pretrained_param_tensor in pretrained_sd.items():
        model_param_tensor = model_sd[layer]
        if model_param_tensor.shape == pretrained_param_tensor.shape:
            model_param_tensor.copy_(pretrained_param_tensor)
        else:  # for the input layer, where shapes do not match
            # set only the last 15 columns of the layer (the one hot cell type columns in the input data tensors)
            model_param_tensor[:, -15:] = pretrained_param_tensor
            model_param_tensor[:, :-15] = 0.0  # initialize other inputs as 0 weight

    model.load_state_dict(model_sd)

def train_test_MLP(input_layer_size, train_dataloader, test_dataloader, num_epochs, lr, dropout, weight_decay, path_to_pretrained):
    model = MLP(input_layer_size, [512, 512, 8], activation_layer=nn.Tanh, dropout=dropout)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    if path_to_pretrained is not None:
        initialize_weights(model=model, path_to_pretrained=path_to_pretrained)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_losses = []
    test_losses = []
    r2_scores = []

    for epoch in range(num_epochs):
        # training
        model.train()
        train_loss = 0.0
        for data_batch, targets_batch in train_dataloader:
            data_batch, targets_batch = data_batch.to(device), targets_batch.to(device)
            optimizer.zero_grad()
            outputs = model(data_batch)
            loss = criterion(outputs, targets_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()*data_batch.size(0)
        train_loss /= len(train_dataloader.sampler)
        train_losses.append(train_loss)

        # testing
        model.eval()
        test_loss = 0.0
        all_outputs = []
        all_targets = []
        with torch.no_grad():
            for data_batch, targets_batch in test_dataloader:
                data_batch, targets_batch = data_batch.to(device), targets_batch.to(device)
                outputs = model(data_batch)
                loss = criterion(outputs, targets_batch)
                test_loss += loss.item()*data_batch.size(0)
                all_outputs.append(outputs)
                all_targets.append(targets_batch)
            test_loss /= len(test_dataloader.sampler)
            test_losses.append(test_loss)

        all_outputs = torch.cat(all_outputs, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        all_outputs, all_targets = all_outputs.cpu(), all_targets.cpu()
        r2_scores.append(r2_score(all_targets, all_outputs, multioutput="raw_values"))
        # r2_scores is a list of num_epochs numpy arrays each 8 long (for each ephys feature)
        
    return model, all_outputs, all_targets, train_losses, test_losses, r2_scores

# for nested cross-validation loop
def make_objective(trainval_idx, dataset, input_layer_size, num_epochs, batch_size, seed, outer_fold, model):
    def objective(trial):
        # hyperparameter space
        lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        inner_scores = []
        networks = []

        labels = cell_embeddings_adata.obs["CellType"].iloc[trainval_idx].values

        for inner_fold, (sub_train_idx, val_idx) in enumerate(inner_cv.split(trainval_idx, labels)):
            sub_train_dataset = Subset(dataset, trainval_idx[sub_train_idx])
            val_dataset = Subset(dataset, trainval_idx[val_idx])
            path_to_pretrained = None if model == "onehot" else f"outerfold{outer_fold}_innerfold{inner_fold}_onehotMLP_weights.pth"

            sub_train_loader = DataLoader(sub_train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

            network, _, _, _, val_losses, r2_scores = train_test_MLP(
                input_layer_size=input_layer_size,
                train_dataloader=sub_train_loader,
                test_dataloader=val_loader,
                num_epochs=num_epochs,
                lr=lr,
                dropout=dropout,
                weight_decay=weight_decay,
                path_to_pretrained=path_to_pretrained
            )
            networks.append(network)
            inner_scores.append(val_losses[-1])
            
        if model == "onehot":
            trial.set_user_attr("networks", networks)
        return np.mean(inner_scores)
        
    return objective


In [4]:
models = ["onehot", "scgpt", "hvg", "ion"]
num_cells = len(hvg_dataset)
input_layer_sizes = [15, 512+15, 512+15, 328+15]
input_layer_sizes = dict(zip(models, input_layer_sizes))

best_hyperparameters = {}
num_epochs = 1500
batch_size = 25

shuffled_cell_indices=[]

results_to_store = {"predictions": [], "train_losses": [], "test_losses": [], "r2_scores": []}
results = {model: deepcopy(results_to_store) for model in models}
results["cell type mean"] = {"predictions": [], "r2_scores": []}

ephys_truth=[]


outer_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
optuna.logging.set_verbosity(optuna.logging.FATAL)
sampler = TPESampler(seed=seed)

for outer_fold, (trainval_idx, test_idx) in enumerate(outer_cv.split(np.arange(num_cells), np.array(cell_embeddings_adata.obs["CellType"]))):
    shuffled_cell_indices.append(test_idx)

    for model in models:
        # HYPERPARAMETER TUNING:
        if model == "onehot":
            objective = make_objective(
                trainval_idx=trainval_idx,
                dataset=datasets[model],
                input_layer_size=input_layer_sizes[model],
                num_epochs=num_epochs,
                batch_size=batch_size,
                seed=seed,
                outer_fold=outer_fold,
                model=model
            )
            study = optuna.create_study(direction="minimize", sampler=sampler)
            study.optimize(objective, n_trials=100, n_jobs=-1, gc_after_trial=True, catch=(ValueError,))
            onehotnetworks = study.best_trial.user_attrs["networks"]
            for inner_fold, onehotnet in enumerate(onehotnetworks):
                onehotnet.cpu()
                statedict_file_name = f"outerfold{outer_fold}_innerfold{inner_fold}_onehotMLP_weights.pth"
                torch.save(onehotnet.state_dict(), statedict_file_name)
            study_file_name = f"Models/outerfold{outer_fold}_onehotMLP_tuningstudy.pkl"

        else:
            objective = make_objective(
                trainval_idx=trainval_idx,
                dataset=datasets[model],
                input_layer_size=input_layer_sizes[model],
                num_epochs=num_epochs,
                batch_size=batch_size,
                seed=seed,
                outer_fold=outer_fold,
                model=model
            )
            study = optuna.create_study(direction="minimize", sampler=sampler)
            study.optimize(objective, n_trials=100, n_jobs=-1, gc_after_trial=True, catch=(ValueError,))
            study_file_name = f"Models/outerfold{outer_fold}_celltypeinitialized_{model}MLP_tuningstudy.pkl"

        joblib.dump(study, study_file_name)
        best_hyperparameters[model] = study.best_params

        # FINAL TRAINING ON FULL TRAINING SET, USING BEST HYPERPARAMETERS:
        train_subset = Subset(datasets[model], trainval_idx)
        test_subset = Subset(datasets[model], test_idx)

        train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
        test_dataloader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

        path_to_pretrained = None if model == "onehot" else f"outerfold{outer_fold}_onehotMLP_weights.pth"
        network, predict_subset, ephys_truth_subset, train_losses, test_losses, r2_scores = train_test_MLP(
            input_layer_size=input_layer_sizes[model],
            train_dataloader=train_dataloader,
            test_dataloader=test_dataloader,
            num_epochs=num_epochs,
            lr=best_hyperparameters[model]["lr"],
            dropout=best_hyperparameters[model]["dropout"],
            weight_decay=best_hyperparameters[model]["weight_decay"],
            path_to_pretrained=path_to_pretrained
        )
        network.cpu()
        statedict_file_name = f"Models/outerfold{outer_fold}_onehotMLP_weights.pth" if model == "onehot" else f"Models/outerfold{outer_fold}_celltypeinitialized_{model}MLP_weights.pth"
        torch.save(network.state_dict(), statedict_file_name)

        # store results
        results[model]["predictions"].append(predict_subset)
        results[model]["train_losses"].append(train_losses)
        results[model]["test_losses"].append(test_losses)
        results[model]["r2_scores"].append(r2_scores)

    ephys_truth.append(ephys_truth_subset)
    print(f"Outer Fold {outer_fold} Best Hyperparameters:")
    for key, value in best_hyperparameters.items():
        print(f"{key}: {value}")
    
    # train/test cell type mean prediction
    trainset_ephys_vals = np.array(ephys_tensor[trainval_idx])
    trainset_cells_types = list(cell_embeddings_adata.obs["CellType"].iloc[trainval_idx])
    trainset_cell_types_to_ephys = {}
    for cells_type, ephys_vals in zip(trainset_cells_types, trainset_ephys_vals):
        if cells_type not in trainset_cell_types_to_ephys:
            trainset_cell_types_to_ephys[cells_type] = []
        trainset_cell_types_to_ephys[cells_type].append(ephys_vals)
    
    for cell_types in trainset_cell_types_to_ephys:
        trainset_cell_types_to_ephys[cell_types] = np.mean(trainset_cell_types_to_ephys[cell_types], axis=0)
    
    celltype_preds_subset = []
    for cells_type in list(cell_embeddings_adata.obs["CellType"].iloc[test_idx]):
        celltype_preds_subset.append(trainset_cell_types_to_ephys[cells_type])
    celltype_preds_subset = np.stack(celltype_preds_subset, axis=0)
    
    results["cell type mean"]["predictions"].append(celltype_preds_subset)
    results["cell type mean"]["r2_scores"].append(r2_score(ephys_truth_subset, celltype_preds_subset, multioutput="raw_values"))

/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Outer Fold 0 Best Hyperparameters:
onehot: {'lr': 0.008619593067260322, 'dropout': 0.011574093996053747, 'weight_decay': 5.619208802319713e-05}
scgpt: {'lr': 0.008138433030095949, 'dropout': 0.008456224462810802, 'weight_decay': 5.229036679144225e-05}
hvg: {'lr': 0.00016243821772804888, 'dropout': 0.0745322771535668, 'weight_decay': 1.0835307598298817e-05}
ion: {'lr': 0.00014988565632426825, 'dropout': 0.022124819604326995, 'weight_decay': 0.004883852017483785}


/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: Runt

Outer Fold 1 Best Hyperparameters:
onehot: {'lr': 0.0170416669416982, 'dropout': 0.005551109177884261, 'weight_decay': 0.0020405687252169178}
scgpt: {'lr': 0.09124239062818061, 'dropout': 0.04736442685787445, 'weight_decay': 0.0009095326907594966}
hvg: {'lr': 0.000130475396146466, 'dropout': 0.015612063074573468, 'weight_decay': 0.0011032608952524173}
ion: {'lr': 0.00010546514758464902, 'dropout': 0.0060535234801404325, 'weight_decay': 0.0022379612357232926}


/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow

Outer Fold 2 Best Hyperparameters:
onehot: {'lr': 0.004022565085083926, 'dropout': 0.0232030885399106, 'weight_decay': 0.000855723155435804}
scgpt: {'lr': 0.009324088345776802, 'dropout': 0.013740525135361892, 'weight_decay': 0.001346431965257474}
hvg: {'lr': 0.00017085627203524743, 'dropout': 0.022565306927752438, 'weight_decay': 2.4712188307816893e-05}
ion: {'lr': 0.00010679330541001998, 'dropout': 0.021364489168056333, 'weight_decay': 0.0011914596112090538}


/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1292: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overf

Outer Fold 3 Best Hyperparameters:
onehot: {'lr': 0.006281811530526401, 'dropout': 0.008803371566246165, 'weight_decay': 0.0022865923643135967}
scgpt: {'lr': 0.0002634115886014501, 'dropout': 0.011920637368633968, 'weight_decay': 0.0019287855182708263}
hvg: {'lr': 0.00011119974271548853, 'dropout': 0.004630756033711436, 'weight_decay': 5.592677822459001e-05}
ion: {'lr': 0.00012140665943994739, 'dropout': 0.006786929923470186, 'weight_decay': 2.3106767811153316e-05}


In [5]:
shuffled_cell_indices = np.concatenate(shuffled_cell_indices)

for model in models:
    results[model]["predictions"] = [tensor.numpy() for tensor in results[model]["predictions"]]
    results[model]["predictions"] = np.vstack(results[model]["predictions"])

    results[model]["train_losses"] = np.array(results[model]["train_losses"])
    results[model]["test_losses"] = np.array(results[model]["test_losses"])
    results[model]["r2_scores"] = np.array(results[model]["r2_scores"])

results["cell type mean"]["predictions"] = np.vstack(results["cell type mean"]["predictions"])

ephys_truth = [tensor.numpy() for tensor in ephys_truth]
ephys_truth = np.vstack(ephys_truth)

In [6]:
np.save('celltypeinitalized_shuffled_cell_indices.npy', shuffled_cell_indices)
with open('celltypeinitalized_results.pkl', 'wb') as f:
    pickle.dump(results, f)
np.save('celltypeinitalized_ephys_truth.npy', ephys_truth)

In [7]:
gc.collect()
torch.cuda.empty_cache()